# 利用者の選択を学習して変換結果を変える

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_japanese_input/kkc_user_history.ipynb)

NLP クックブック 27-09 ／ 日本語入力 (かな漢字変換)

一度選び直した変換は、次から最初に出てほしい。利用者ごとの入力履歴を使って、変換の優先順位を変えたい。

In [1]:
!pip install -q fugashi unidic-lite jaconv marisa-trie

In [2]:
import io, math, re, urllib.request, zipfile
from collections import Counter, defaultdict
import jaconv
import marisa_trie
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/000148/files/"
dic, unigram, bigram = defaultdict(set), Counter(), Counter()
for f in ["789_ruby_5639.zip", "752_ruby_2438.zip", "773_ruby_5968.zip"]:
    for s in re.split(r"(?<=。)|\n", aozora(base + f)):
        if not s.strip():
            continue
        prev = "<s>"; unigram[prev] += 1
        for w in tagger(s.strip()):
            surface, reading = w.surface, jaconv.kata2hira(w.feature.kana or w.surface)
            dic[reading].add(surface); unigram[surface] += 1; bigram[prev, surface] += 1; prev = surface
        bigram[prev, "</s>"] += 1
total = sum(unigram.values())

In [3]:
class Converter:
    """bigram によるかな漢字変換 + 利用者の履歴 (確定した単語と、その並び) による補正"""
    def __init__(self, history_weight=3.0):
        self.user_dic = defaultdict(set)          # 利用者が登録した語: 読み → 表記
        self.user_uni, self.user_bi = Counter(), Counter()
        self.history_weight = history_weight
        self.rebuild()

    def rebuild(self):
        self.trie = marisa_trie.Trie(set(dic) | set(self.user_dic))

    def cost(self, prev, word, lam=0.8):
        p = lam * (bigram.get((prev, word), 0) / unigram[prev] if unigram.get(prev) else 0.0) + (1 - lam) * unigram.get(word, 0) / total
        base = -math.log(p) if p > 0 else 20.0
        # 履歴にある単語・並びは、使った回数に応じてコストを下げる (回数の対数に比例)
        bonus = self.history_weight * (math.log1p(self.user_uni[word]) + math.log1p(self.user_bi[prev, word]))
        return max(base - bonus, 0.1)

    def convert(self, kana):
        best = [dict() for _ in range(len(kana) + 1)]
        best[0]["<s>"] = (0.0, None, None)
        for i in range(len(kana)):
            cands = [(i + len(r), s) for r in self.trie.prefixes(kana[i:]) for s in dic.get(r, set()) | self.user_dic.get(r, set())] + [(i + 1, kana[i])]
            for j, word in cands:
                for prev, (c, _, _) in best[i].items():
                    new = c + self.cost(prev, word)
                    if word not in best[j] or new < best[j][word][0]:
                        best[j][word] = (new, i, prev)
        word = min(best[-1], key=lambda w: best[-1][w][0] + self.cost(w, "</s>"))
        words, pos = [], len(kana)
        while pos > 0:
            _, prev_pos, prev = best[pos][word]
            words.append(word); pos, word = prev_pos, prev
        return words[::-1]

    def commit(self, words):
        """利用者が確定した単語列を履歴に記録する"""
        for prev, word in zip(["<s>"] + words, words):
            self.user_uni[word] += 1; self.user_bi[prev, word] += 1

    def forget(self):
        """履歴をすべて消す (登録した単語は残す)"""
        self.user_uni.clear(); self.user_bi.clear()

    def register(self, reading, surface):
        """単語登録: 辞書にない語を追加する"""
        self.user_dic[reading].add(surface); self.user_uni[surface] += 1; self.rebuild()

conv = Converter()
tests = ["きかいがない", "きかいをうかがう", "よいきかいだ"]
print("学習前:", ["".join(conv.convert(k)) for k in tests])
for n in range(1, 4):
    conv.commit(["機械", "が", "ない"])             # 利用者が「機会」を「機械」に選び直して確定した
    print(f"{n} 回確定した後:", ["".join(conv.convert(k)) for k in tests])

学習前: ['機会がない', '機会を窺う', '好い機会だ']
1 回確定した後: ['機会がない', '機械を窺う', '好い機会だ']
2 回確定した後: ['機械がない', '機械を窺う', '好い機会だ']
3 回確定した後: ['機械がない', '機械を窺う', '好い機会だ']


In [4]:
# 単語登録: 辞書にない固有名詞や新しい語
tests = ["すまほをみる", "ちゃっとぼっととはなす"]
print("登録前:", ["".join(conv.convert(k)) for k in tests])
conv.register("すまほ", "スマホ")
conv.register("ちゃっとぼっと", "チャットボット")
print("登録後:", ["".join(conv.convert(k)) for k in tests])

登録前: ['済ま歩を見る', '茶っと坊っとと話す']
登録後: ['スマホを見る', 'チャットボットと話す']


In [5]:
# 履歴の重みが強すぎると、関係のない文脈まで巻き込んで変わってしまう
for weight in [1.0, 3.0, 10.0]:
    c = Converter(history_weight=weight)
    for _ in range(3):
        c.commit(["機械", "が", "ない"])
    print(f"重み {weight}:", ["".join(c.convert(k)) for k in ["きかいがない", "きかいをうかがう", "よいきかいだ"]])

conv.forget()
print("履歴を消した後:", "".join(conv.convert("きかいがない")))

重み 1.0: ['機会がない', '機会を窺う', '好い機会だ']
重み 3.0: ['機械がない', '機械を窺う', '好い機会だ']
重み 10.0: ['機械がない', '機械を窺う', '好い機械だ']
履歴を消した後: 機会がない


- 履歴の学習には、単語の頻度 (この語をよく使う) と、単語の並び (この語の後ろではこの語) の 2 種類がある。並びまで覚えると、文脈に応じた学習になる。
- 学習が強すぎると、一度選んだ表記が関係のない文脈でも出るようになる (過学習)。回数の対数に比例させる、古い履歴の重みを徐々に下げる、といった調整をする。
- 「学習してほしくない誤変換」を確定してしまうこともある。履歴を取り消す手段 (直前の学習を忘れる、履歴の消去) が必要になる。
- 履歴は個人情報のかたまりなので、端末の中だけで扱い、外に送らないのが基本。